In [ ]:
# Cell 0 - settings for this run. The only cell to edit between runs.
import re

RUN_NAME = "run2"        # used in file names; letters, digits, - and _ only
TARGET = "log"           # "price" = predict dollars (run1), "log" = predict log(price)
EPOCHS = 6               # run1's val error stopped improving by about epoch 4
COMPARE_WITH = "auto"    # earlier run to score side by side: "auto" = the one attached
                         # input that has checkpoints/best.pt; or a path; or None
COMPARE_NAME = "run1"    # label for that earlier run in the tables

assert re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME) and re.fullmatch(r"[A-Za-z0-9_-]+", COMPARE_NAME)
assert TARGET in ("price", "log")
print(f"{RUN_NAME}: target={TARGET}, epochs={EPOCHS}, compare with {COMPARE_NAME} ({COMPARE_WITH})")

In [ ]:
# Cell 1 - check the GPU actually works
import torch

print("torch", torch.__version__)
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU T4 x2, then restart the session.")
print("GPU:", torch.cuda.get_device_name(0))

# Run a real convolution on the GPU. If this torch build can't drive the card
# (e.g. an old P100), it fails here in seconds instead of minutes later.
x = torch.randn(8, 3, 224, 224, device="cuda")
w = torch.randn(16, 3, 3, 3, device="cuda")
y = torch.nn.functional.conv2d(x, w)
torch.cuda.synchronize()
print("GPU works, test output shape:", tuple(y.shape))

In [ ]:
# Cell 2 - helper, code, timm
import os
import subprocess
from shlex import quote as q  # quotes paths for the shell


def run(cmd):
    """Run a shell command, show its output live, and STOP the notebook if it fails.

    A plain `!command` ignores failures, so a broken smoke test would still
    carry on into the long run. This raises instead.
    """
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, executable="/bin/bash")
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed (exit code {p.returncode}): {cmd}")


REPO = "/tmp/repo"
run(f"rm -rf {REPO} && git clone -q -b feature/train-resume --depth 1 "
    f"https://github.com/Akash7762/Amazon-Vision-Based-Price-Predictor.git {REPO}")
os.chdir(REPO)
run("git log --oneline -1")
run("grep -q -- '--select-by' model/train.py && test -f model/evaluate.py && echo 'code is up to date' "
    "|| (echo 'OLD CODE on GitHub: run git push on your PC first' && exit 1)")

run('pip install -q "timm>=1.0.27"')
# pip must not have swapped Kaggle's GPU torch for a CPU-only one
run('python -c "import torch, timm; assert torch.cuda.is_available(); '
    'print(\'torch\', torch.__version__, \'| timm\', timm.__version__, \'| cuda OK\')"')

In [ ]:
# Cell 3 - point at the data (and the earlier run), and check it
import glob
import os
import pandas as pd

META = "/kaggle/input/datasets/akashverma7762/amazon-price-metadata/metadata.csv"
IMG = "/kaggle/input/datasets/akashverma7762/amazon-price-images-v2/images"

if not (os.path.exists(META) and os.path.isdir(IMG)):
    # Fallback search. It only looks at dataset-folder level and never lists the
    # 73k-image folder itself, which is what made the old search take minutes.
    metas = glob.glob("/kaggle/input/*/metadata.csv") + glob.glob("/kaggle/input/datasets/*/*/metadata.csv")
    imgs = glob.glob("/kaggle/input/*/images") + glob.glob("/kaggle/input/datasets/*/*/images")
    print("found metadata:", metas)
    print("found images:  ", imgs)
    assert len(metas) == 1 and len(imgs) == 1, (
        "Attach exactly one metadata and one images dataset: amazon-price-metadata and "
        "amazon-price-images-v2 (not the old amazon-price-images).")
    META, IMG = metas[0], imgs[0]

ids = set(pd.read_csv(META)["sample_id"].astype(str))
have = {f[:-4] for f in os.listdir(IMG) if f.endswith(".jpg")}  # takes ~1 min, the folder is big
print(f"META = {META}")
print(f"IMG  = {IMG}")
print(f"images={len(have)} missing={len(ids - have)}")
assert len(ids - have) < 100, "Images missing - wrong images dataset attached?"

manifest = os.path.join(IMG, "_preprocessing.json")
print(open(manifest).read() if os.path.exists(manifest) else "WARNING: no _preprocessing.json")

CMP_CKPT = None
if COMPARE_WITH == "auto":
    # <input>/.../checkpoints/best.pt up to 4 folders deep. Not deeper on purpose:
    # that would list the 73k-image folder.
    hits = sorted({h for depth in range(1, 5)
                   for h in glob.glob("/kaggle/input/" + "*/" * depth + "checkpoints/best.pt")})
    print("earlier-run checkpoints found:", hits)
    assert len(hits) == 1, (
        f"Expected exactly one attached earlier run with checkpoints/best.pt, found {len(hits)}. "
        f"Attach {COMPARE_NAME}'s output (the amazon-price-run1 dataset), "
        f"or set COMPARE_WITH in Cell 0 to a path or to None.")
    CMP_CKPT = hits[0]
elif COMPARE_WITH:
    assert os.path.exists(COMPARE_WITH), f"COMPARE_WITH not found: {COMPARE_WITH}"
    CMP_CKPT = COMPARE_WITH
print("compare with:", CMP_CKPT)

In [ ]:
# Cell 4 - score the earlier run on val with the same script (~2 min).
# Done before training so a problem with the evaluation shows up now, not after the long run.
EVAL = "/kaggle/working/eval"
if CMP_CKPT:
    run(f"python -u model/evaluate.py --checkpoint {q(CMP_CKPT)} --name {COMPARE_NAME} "
        f"--metadata {q(META)} --image-dir {q(IMG)} --split val --out-dir {q(EVAL)}")
else:
    print("COMPARE_WITH is None - skipping")

In [ ]:
# Cell 5 - smoke test: 5 batches with this run's settings, ~1-2 min.
run(f"python -u model/train.py --subset-mode full --metadata {q(META)} --image-dir {q(IMG)} "
    f"--target {TARGET} --epochs 1 --max-batches 5 --batch-size 64 --num-workers 4 "
    f"--checkpoint-dir /tmp/smoke --log-path /tmp/smoke/smoke.log")

In [ ]:
# Cell 6 - the real training run (~15 min per epoch on a T4)
# --select-by val_mae: best.pt is the epoch with the lowest val error in dollars,
#   the number runs are compared on (the loss isn't comparable across targets).
# --max-hours 8: stops cleanly before an epoch that would go past 8 hours, so the
#   run always finishes and saves its output instead of being killed by Kaggle.
run(f"python -u model/train.py --subset-mode full --metadata {q(META)} --image-dir {q(IMG)} "
    f"--target {TARGET} --select-by val_mae --epochs {EPOCHS} --batch-size 64 --lr 1e-4 "
    f"--num-workers 4 --max-hours 8 "
    f"--checkpoint-dir /kaggle/working/checkpoints --log-path /kaggle/working/logs/train.log")

In [ ]:
# Cell 7 - score this run's best.pt on val, then compare with the earlier run (~2 min)
run(f"python -u model/evaluate.py --checkpoint /kaggle/working/checkpoints/best.pt "
    f"--name {RUN_NAME} --metadata {q(META)} --image-dir {q(IMG)} --split val --out-dir {q(EVAL)}")
if CMP_CKPT:
    run(f"python -u model/evaluate.py --compare {q(EVAL + '/' + COMPARE_NAME + '_val.json')} "
        f"{q(EVAL + '/' + RUN_NAME + '_val.json')}")

In [ ]:
# Cell 8 - summary, trim checkpoints, plot
import glob
import json
import os
import re
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

CK = "/kaggle/working/checkpoints"
h = json.load(open(f"{CK}/history.json"))
ev = json.load(open(f"{EVAL}/{RUN_NAME}_val.json"))
base_mae = ev["baseline_median"]["mae"]
cmp_mae = json.load(open(f"{EVAL}/{COMPARE_NAME}_val.json"))["model"]["mae"] if CMP_CKPT else None

best = min(h, key=lambda r: r["val_mae"])
print(f"epochs done: {h[-1]['epoch']} of {EPOCHS}")
if h[-1]["epoch"] < EPOCHS:
    print(f"Stopped early by the time budget - resume from epoch_{h[-1]['epoch']}.pt")
print(f"best.pt = epoch {best['epoch']}: val MAE ${ev['model']['mae']:.3f} "
      f"(training logged ${best['val_mae']:.3f} for that epoch - the two should agree closely)")
print(f"median baseline ${base_mae:.3f}" + (f" | {COMPARE_NAME} ${cmp_mae:.3f}" if cmp_mae else ""))

# Each checkpoint is ~330 MB. Keep best.pt and the newest epoch_N.pt (for resuming).
done = sorted(int(re.search(r"epoch_(\d+)\.pt$", p).group(1)) for p in glob.glob(f"{CK}/epoch_*.pt"))
for e in done[:-1]:
    os.remove(f"{CK}/epoch_{e}.pt")

ep = [r["epoch"] for r in h]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ep, [r["train_loss"] for r in h], marker="o", label="train")
ax1.plot(ep, [r["val_loss"] for r in h], marker="o", label="val")
ax1.set(title=f"{RUN_NAME} loss (target={TARGET})", xlabel="epoch", ylabel="SmoothL1 loss")
ax2.plot(ep, [r["val_mae"] for r in h], marker="o", c="C1", label=f"{RUN_NAME} val")
ax2.axhline(base_mae, ls="--", c="gray", label="median-price baseline")
if cmp_mae:
    ax2.axhline(cmp_mae, ls=":", c="C0", label=f"{COMPARE_NAME} best.pt")
ax2.set(title="val error in dollars", xlabel="epoch", ylabel="MAE ($)")
for ax in (ax1, ax2):
    ax.legend(); ax.grid(alpha=0.3); ax.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.savefig("/kaggle/working/loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nSaved output:")
for f in sorted(glob.glob("/kaggle/working/**", recursive=True)):
    if os.path.isfile(f):
        print(f"{os.path.getsize(f) / 1e6:8.1f} MB  {f}")
print(json.dumps(h, indent=1))